# Scenario Gold — Monte Carlo and serving

Run the retained calculation engine on Gold inputs. Publish scenario summaries and probability curves separately from source profiles. The JSON serving projection drives the public dashboard without granting it database credentials. This small demo collects bounded data to the notebook driver; it is not distributed Monte Carlo.


In [ ]:
from pathlib import Path
import json, os, sys
root_hint = os.environ.get("AIDP_PROJECT_ROOT")
if root_hint:
    root = Path(root_hint).resolve()
else:
    root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "race_demo/pipeline.py").is_file()), None)
if root is None or not (root / "race_demo/pipeline.py").is_file():
    raise RuntimeError("Set AIDP_PROJECT_ROOT to this repository checkout.")
sys.path.insert(0, str(root))
from race_demo.pipeline import generate_bronze, build_silver, build_gold, serving_document, write_once, digest
from race_demo.workbench import publish_table, table_name
execution = os.environ.get("RACE_DEMO_EXECUTION", "LOCAL_DEMO")
if execution not in {"LOCAL_DEMO", "AIDP_WORKBENCH"}:
    raise ValueError("Unknown execution mode")
if execution == "AIDP_WORKBENCH":
    if "spark" not in globals():
        raise RuntimeError("Use an AIDP notebook runtime with Spark/Delta configured.")
    table_name("bronze")  # Validate the dedicated demo namespace before writes.
    if not os.environ.get("RACE_DEMO_OUTPUT"):
        raise RuntimeError("Set RACE_DEMO_OUTPUT to a new writable demo volume directory.")
output = Path(os.environ.get("RACE_DEMO_OUTPUT", str(root / "build/notebooks")))
print("Execution:", execution, "| Evidence: SYNTHETIC_DEMO")


In [ ]:
bronze = generate_bronze()
silver = build_silver(bronze)
if execution == "AIDP_WORKBENCH":
    profiles = [r.asDict() for r in spark.table(table_name("gold_profiles")).orderBy("driver_id").collect()]
    deviations = [r["deviation_seconds"] for r in spark.table(table_name("gold_deviations")).orderBy("sample_index").collect()]
    gold = {"profiles": profiles, "deviations": deviations}
else:
    gold = json.loads((output / "gold/profiles.json").read_text())
document = serving_document(bronze, silver, gold, execution=execution)
if execution == "AIDP_WORKBENCH":
    summaries = [{"field_size": s["field_size"], "simulations": s["simulations"], "probability": s["full_field_probability"], "seed": s["simulation_seed"], "evidence_type": "SYNTHETIC_DEMO"} for s in document["scenarios"].values()]
    curves = [{"field_size": s["field_size"], "delay_seconds": p["delaySeconds"], "probability": p["catchProbability"], "evidence_type": "SYNTHETIC_DEMO"} for s in document["scenarios"].values() for p in s["curve"]]
    print(publish_table(spark, "gold_scenarios", summaries))
    print(publish_table(spark, "gold_curve", curves))
write_once(output / "serving/demo.json", document)
print({"execution": execution, "serving_sha256": digest(document), "status": "READ_BACK_VERIFIED"})
